# Vision · 범위 제한 기반 빈칸·부분 구현 문제

원본에 마스킹이 없어 **완성된 실습 코드를 바탕으로 새로 선정한 연습 문제**입니다. 실제 시험 문항이나 공식 배점을 의미하지 않습니다.

- ResNet: **0~5절**, 전체 셀15까지. 이번 요청의 “Chapter 6 이전”을 적용하여 6절 자체도 제외합니다.
- ViT: **6절 시작 전**, 전체 셀23까지. 원본 절3/4 순서가 뒤섞여 있어 셀 번호도 함께 기록합니다.
- DETR: 기존 확정 범위인 **13절 포함**, 전체 셀30까지.
- U-Net: 지정 노트북 전체. 설치·다운로드·긴 학습·그래프 꾸미기는 제외합니다.

근거: [시험 출제 포인트 PDF](../../AI인증시험_출제포인트.pdf) 1쪽 Vision의 레이어 이해, 예측/정답을 학습으로 연결, 데이터 구성, 이미지 변환 라이브러리, 평가 Metric. 문항별 셀 번호는 **Markdown 포함,1부터**입니다.

**사용법:** 준비 셀을 먼저 실행하고 각 문제에서 `MISSING` 또는 `raise NotImplementedError`를 올바른 코드로 교체 → 바로 아래 테스트 실행 → 실패하면 shape/축/순서 확인 → 마지막의 접힌 해설 확인. `# ????`는 원본에서 가져온 빈칸이 아니라 새 마스킹 표시입니다. 각 문제는 독립되어 이전 문제의 답이 필요 없습니다. 답을 수정하면 함수 셀을 먼저 다시 실행하세요. 예상 5~10분/문제(연습용 가안). 작성한 이 노트북을 제출하면 채점할 수 있습니다.

CPU·PyTorch·torchvision·einops·NumPy·Pillow만 사용합니다. 모델/데이터 다운로드 없음. 테스트는 shape뿐 아니라 수치, 학습 갱신, 잘못된 축, 다른 인자도 검사합니다. 통과는 작은 입력 검증이며 실전 정확도나 독립 숙달을 보장하지 않습니다.

**원본과 차이:** ViT FFN은 실제 코드의 ReLU 사용(주석의 GELU와 다름). U-Net DoubleConv는 실제 코드처럼 BN 없음. DETR attention reshape는 query/key 축이 뒤바뀐 설명·슬라이스를 교정하고 문제에서 명시합니다. ResNet AMP는 CPU에서 비활성 scaler로 순서를 확인합니다. 설치된 PyTorch 2.5.1/torchvision 0.20.1/einops 0.8.2에서 검증합니다.

이미지 텐서는 PyTorch의 실제 축 순서 **B,C,H,W**로 표기합니다. W,H를 구분하는 비정사각형 입력도 검사합니다. 이해가 안 되는 문제 번호·줄을 알려주면 작은 행렬 예시로 다시 설명할 수 있습니다.


In [ ]:
import copy
import math
import numpy as np
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import torchvision
import torchvision.transforms as T  # T = torchvision.transforms
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from einops import rearrange, repeat
from einops.layers.torch import Rearrange

torch.set_num_threads(1)
MISSING = object()  # ???? 대신 실행 가능한 표식. 이 값을 올바른 코드로 바꾸세요.
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2023, 0.1994, 0.2010)

def close(actual, expected):
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)

class ModeProbe(nn.Module):
    # train/eval과 gradient 모드를 기록하는 작은 제공 분류기입니다.
    def __init__(self, width=4, classes=3):
        super().__init__()
        self.fc = nn.Linear(width, classes)
        self.seen = []
    def forward(self, x):
        self.seen.append((self.training, torch.is_grad_enabled()))
        return self.fc(x)


## U01. 이미지와 binary mask의 채널축

**PDF 점검:** 데이터 구성 · **근거:** [04_Unet.ipynb](../../vision/04_Unet.ipynb) 전체 셀 5

제공 NumPy float32 img/mask=(H,W)를 각각 (1,H,W) PyTorch 텐서로 바꾸세요. mask는 이미0/1이며 클래스 인덱스 long으로 바꾸지 않습니다. 반환 img,mask.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_u01(img,mask):
    img=MISSING
    mask=MISSING
    return MISSING,MISSING

In [ ]:
def check_u01(fn):
    for h,w in [(4,5),(7,6)]:
        mask=(np.arange(h*w).reshape(h,w)%2).astype(np.float32);img=mask*.8
        a,b=fn(img,mask)
        close(a,torch.tensor(img)[None]);close(b,torch.tensor(mask)[None])
        assert a.dtype==b.dtype==torch.float32
        assert torch.stack([b,b]).shape==(2,1,h,w)

try:
    check_u01(solve_u01)
    print("PASS · U01")
except Exception as error:
    print("FAIL · U01 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: mask(4,4)→(1,4,4)→배치2장(2,1,4,4)。BCEWithLogitsLoss에서 prediction과 mask는 같은 shape의 float입니다. mask1은 전경,0은 배경입니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

def solve_u01(img,mask):
    img=np.expand_dims(img,axis=0)
    mask=np.expand_dims(mask,axis=0)
    return torch.from_numpy(img),torch.from_numpy(mask)
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## U02. DoubleConv의 padding과 채널

**PDF 점검:** 레이어 구성·동작 · **근거:** [04_Unet.ipynb](../../vision/04_Unet.ipynb) 전체 셀 7

Conv3×3(padding1,bias=False)→ReLU(inplace=True)를 두 번 구성하세요. 채널은 in→mid→out입니다. 원본 실제 DoubleConv에는 BN이 없습니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_u02(in_channels,mid_channels,out_channels):
    return nn.Sequential(
        MISSING,
        nn.ReLU(inplace=True),
        MISSING,
        nn.ReLU(inplace=True),
    )

In [ ]:
def check_u02(fn):
    for ci,cm,co in [(1,3,2),(2,4,3)]:
        m=fn(ci,cm,co)
        assert [type(t) for t in m]==[nn.Conv2d,nn.ReLU,nn.Conv2d,nn.ReLU]
        assert m[0].bias is None and m[2].bias is None
        assert m[1].inplace and m[3].inplace
        assert m[0].out_channels==cm and m[2].out_channels==co
        x=torch.randn(2,ci,7,9);got=m(x)
        expected=F.relu(F.conv2d(F.relu(F.conv2d(x,m[0].weight,padding=1)),m[2].weight,padding=1))
        close(got,expected);assert got.shape==(2,co,7,9)

try:
    check_u02(solve_u02)
    print("PASS · U02")
except Exception as error:
    print("FAIL · U02 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: (2,1,7,9)→Conv(1,3)→(2,3,7,9)→Conv(3,2)→(2,2,7,9)。kernel3,padding1,stride1이므로 공간 크기는 유지됩니다. U-Net 이름만 보고 BN을 추가하지 않습니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

def solve_u02(in_channels,mid_channels,out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels,mid_channels,3,padding=1,bias=False),
        nn.ReLU(inplace=True),
        nn.Conv2d(mid_channels,out_channels,3,padding=1,bias=False),
        nn.ReLU(inplace=True),
    )
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## U03. Down: pooling 후 convolution

**PDF 점검:** 레이어 구성·동작 · **근거:** [04_Unet.ipynb](../../vision/04_Unet.ipynb) 전체 셀 7

Down.forward입니다. MaxPool2d(2) 후 제공 conv를 적용하세요. conv는 공간 크기를 유지하고 채널만 변경합니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_u03(x,conv):
    x=MISSING
    return MISSING

In [ ]:
def check_u03(fn):
    for h,w in [(6,8),(7,9)]:
        x=torch.arange(h*w).float().reshape(1,1,h,w);conv=nn.Conv2d(1,3,1)
        expected=torch.stack([torch.stack([x[:,:,y:y+2,z:z+2].amax((-2,-1)) for z in range(0,w-1,2)],-1) for y in range(0,h-1,2)],-2)
        close(fn(x,conv),conv(expected));assert fn(x,conv).shape==(1,3,h//2,w//2)

try:
    check_u03(solve_u03)
    print("PASS · U03")
except Exception as error:
    print("FAIL · U03 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: (1,1,7,9)→MaxPool2d2→(1,1,3,4)→conv→(1,3,3,4)。홀수 크기는 내림되므로 나중에 Up에서 padding이 필요할 수 있습니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

def solve_u03(x,conv):
    x=F.max_pool2d(x,kernel_size=2)
    return conv(x)
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## U04. 두 업샘플링 방식

**PDF 점검:** 레이어 구성·동작 · **근거:** [04_Unet.ipynb](../../vision/04_Unet.ipynb) 전체 셀 7

bilinear=True이면 Upsample(scale_factor=2,mode="bilinear",align_corners=True), False이면 ConvTranspose2d(in_channels,in_channels//2,kernel_size=2,stride=2)를 반환하세요.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_u04(in_channels,bilinear):
    if bilinear:
        return MISSING
    return MISSING

In [ ]:
def check_u04(fn):
    for c in (4,6):
        x=torch.randn(2,c,3,5)
        up=fn(c,True);assert isinstance(up,nn.Upsample)
        close(up(x),F.interpolate(x,scale_factor=2,mode='bilinear',align_corners=True))
        assert up(x).shape==(2,c,6,10) and len(list(up.parameters()))==0
        deconv=fn(c,False);assert isinstance(deconv,nn.ConvTranspose2d)
        assert deconv.kernel_size==(2,2) and deconv.stride==(2,2)
        assert deconv(x).shape==(2,c//2,6,10)

try:
    check_u04(solve_u04)
    print("PASS · U04")
except Exception as error:
    print("FAIL · U04 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: (2,4,3,5)→bilinear이면(2,4,6,10),transposed conv이면(2,2,6,10)。전자는 채널 유지·학습 파라미터 없음, 후자는 이 설정에서 채널 절반·학습 가능한 weight가 있습니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

def solve_u04(in_channels,bilinear):
    if bilinear:
        return nn.Upsample(scale_factor=2,mode="bilinear",align_corners=True)
    return nn.ConvTranspose2d(in_channels,in_channels//2,kernel_size=2,stride=2)
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## U05. Up: 크기 정렬과 skip concat

**PDF 점검:** 레이어 구성·동작 · **근거:** [04_Unet.ipynb](../../vision/04_Unet.ipynb) 전체 셀 7

업샘플링은 이미 완료된 x1=(B,Cup,h,w)를 제공합니다. x2=(B,Cskip,H,W)는 skip입니다. 부족한 크기를 좌우/상하에 나누어 padding하고 [x2,x1] 순서로 채널축 cat 후 제공 conv에 넣으세요. diff는0 이상입니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_u05(x1,x2,conv):
    dy=x2.size(2)-x1.size(2)
    dx=x2.size(3)-x1.size(3)
    x1=MISSING
    x=MISSING
    return conv(x)

In [ ]:
def check_u05(fn):
    for dy,dx in [(1,3),(0,0),(2,1)]:
        x1=torch.arange(2*3*4).float().reshape(1,2,3,4);x2=torch.full((1,3,3+dy,4+dx),-1.)
        padded=torch.zeros(1,2,3+dy,4+dx)
        padded[:,:,dy//2:dy//2+3,dx//2:dx//2+4]=x1
        expected=torch.cat([x2,padded],1)
        close(fn(x1,x2,nn.Identity()),expected)
        conv=nn.Conv2d(5,2,1);close(fn(x1,x2,conv),conv(expected))

try:
    check_u05(solve_u05)
    print("PASS · U05")
except Exception as error:
    print("FAIL · U05 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: x1(1,2,3,4),skip(1,3,4,7)→padding은[왼1,오른2,위0,아래1]→cat(1,5,4,7)。ResNet skip은 덧셈, 여기의 U-Net skip은 채널 연결입니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

def solve_u05(x1,x2,conv):
    dy=x2.size(2)-x1.size(2)
    dx=x2.size(3)-x1.size(3)
    x1=F.pad(x1,[dx//2,dx-dx//2,dy//2,dy-dy//2])
    x=torch.cat([x2,x1],dim=1)
    return conv(x)
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## U06. Decoder에 올바른 skip 전달

**PDF 점검:** 레이어 구성·동작 · **근거:** [04_Unet.ipynb](../../vision/04_Unet.ipynb) 전체 셀 7

Encoder의 x1,x2,x3,x4,x5가 이미 계산되어 있습니다. model.up1~up4와 outc를 이용해 bottleneck x5에서 깊은 skip부터 연결하세요. 전체 encoder를 재작성하지 않습니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_u06(model,x1,x2,x3,x4,x5):
    x=model.up1(x5,MISSING)
    x=model.up2(x,MISSING)
    x=model.up3(x,MISSING)
    x=model.up4(x,MISSING)
    return MISSING

In [ ]:
def check_u06(fn):
    class Join(nn.Module):
        def forward(self,a,b):
            assert b.shape[-2]==a.shape[-2]*2
            return F.interpolate(a,size=b.shape[-2:],mode='nearest')+b
    class Decoder(nn.Module):
        def __init__(self):
            super().__init__();self.up1=Join();self.up2=Join();self.up3=Join();self.up4=Join();self.outc=nn.Conv2d(1,2,1)
    for base in (1.,2.):
        xs=[torch.full((1,1,s,s),base*(i+1)) for i,s in enumerate([16,8,4,2,1])]
        m=Decoder();got=fn(m,*xs)
        close(got,m.outc(torch.full((1,1,16,16),base*15)))

try:
    check_u06(solve_u06)
    print("PASS · U06")
except Exception as error:
    print("FAIL · U06 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: x5의1×1→up1+skip x4의2×2→x3의4×4→x2의8×8→x1의16×16。테스트 Join은 연결 순서만 확인하는 작은 대역이며 실제 U-Net 대신 정확도를 평가하지 않습니다. 실제 concat은 앞 문제에서 검증합니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

def solve_u06(model,x1,x2,x3,x4,x5):
    x=model.up1(x5,x4)
    x=model.up2(x,x3)
    x=model.up3(x,x2)
    x=model.up4(x,x1)
    return model.outc(x)
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## U07. 픽셀 logits와 정답 mask로 학습

**PDF 점검:** 예측·정답을 학습으로 연결 · **근거:** [04_Unet.ipynb](../../vision/04_Unet.ipynb) 전체 셀 9, 11

binary segmentation 한 배치를 학습하세요. logits와 masks는 모두 (B,1,H,W), masks는 float0/1입니다. BCEWithLogitsLoss를 선택하고 forward→loss→backward→step을 완성하세요. loss 전에 sigmoid를 붙이지 않습니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_u07(model,optimizer,images,masks):
    model.train()
    optimizer.zero_grad()
    criterion=MISSING
    logits=model(images)
    loss=MISSING
    raise NotImplementedError("???? 이 줄을 완성하세요")
    raise NotImplementedError("???? 이 줄을 완성하세요")
    return loss.item()

In [ ]:
def check_u07(fn):
    for seed in (5,9):
        torch.manual_seed(seed);a=nn.Conv2d(1,1,1);b=copy.deepcopy(a)
        oa=torch.optim.SGD(a.parameters(),lr=.05);ob=torch.optim.SGD(b.parameters(),lr=.05)
        for h,w in [(3,4),(4,5)]:
            x=torch.randn(2,1,h,w);target=(x>0).float()
            ob.zero_grad();loss=F.binary_cross_entropy_with_logits(b(x),target);loss.backward();ob.step()
            assert abs(fn(a,oa,x,target)-loss.item())<1e-6
            for p,q in zip(a.parameters(),b.parameters()):close(p,q)

try:
    check_u07(solve_u07)
    print("PASS · U07")
except Exception as error:
    print("FAIL · U07 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: 한 픽셀 logit0→확률.5,정답1이면 loss≈.693。이 연산을 모든 픽셀에 적용합니다. 이미지 분류의 정답(B,)과 달리 분할 정답은 픽셀마다 있습니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

def solve_u07(model,optimizer,images,masks):
    model.train()
    optimizer.zero_grad()
    criterion=nn.BCEWithLogitsLoss()
    logits=model(images)
    loss=criterion(logits,masks)
    loss.backward()
    optimizer.step()
    return loss.item()
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>

## U08. 확률을 binary segmentation mask로

**PDF 점검:** 모델 출력 활용·데이터 구성 · **근거:** [04_Unet.ipynb](../../vision/04_Unet.ipynb) 전체 셀 15

logits=(B,1,H,W)에 sigmoid 후 threshold 초과인 픽셀을1,나머지를0으로 바꿔 float 텐서로 반환하세요. 원본의0.5를 함수 인자로 일반화했습니다.

완료 기준: 아래 테스트에서 PASS. 해설을 닫고 입력→출력 shape와 빈칸 이유도 말로 설명해 보세요.

In [ ]:
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

# ???? 표시: MISSING 또는 NotImplementedError를 교체하세요.
def solve_u08(logits,threshold):
    probs=MISSING
    return MISSING

In [ ]:
def check_u08(fn):
    probs=torch.tensor([[[[.1,.5,.8],[.6,.2,.95]]]])
    logits=torch.logit(probs)
    for threshold in (.5,.75):
        got=fn(logits,threshold)
        close(got,(probs>threshold).float())
        assert got.shape==(1,1,2,3) and got.dtype==torch.float32

try:
    check_u08(solve_u08)
    print("PASS · U08")
except Exception as error:
    print("FAIL · U08 ·",type(error).__name__,str(error))


<details><summary>시도 후 펼치기: 쉬운 설명 · 예시 · 정답</summary>

예: 확률[.2,.5,.8],threshold=.5→mask[0,0,1]。원본은 >=가 아니라 >입니다. 한 채널 sigmoid이므로 채널축 argmax를 하면 모든 픽셀이0이 되는 잘못된 결과가 나옵니다.

```python
import torch
import torch.nn as nn  # nn = torch.nn
import torch.nn.functional as F  # F = torch.nn.functional
import numpy as np

def solve_u08(logits,threshold):
    probs=torch.sigmoid(logits)
    return (probs > threshold).float()
```

다음 복습: 해설 없이 다시 작성하고, 배치/채널/클래스 수가 바뀌어도 동작하는 이유를 설명하세요.
</details>